# Monte Carlo Convergence Study

This notebook compares three variance-reduction strategies for pricing a European ATM call option:

| Method | Sampler | Expected Convergence Rate |
|---|---|---|
| **Standard** | Pseudo-random (Mersenne Twister) | $O(N^{-1/2})$ |
| **Antithetic** | Paired $Z$ and $-Z$ variates | $O(N^{-1/2})$, smaller constant |
| **Sobol (QMC)** | Scrambled Sobol low-discrepancy | $O((\log N)^d / N)$ |

Reference price: Black-Scholes-Merton closed-form solution.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import lognorm

from src.base_simulator import SimulatorConfig
from src.stochastic_processes import GBMSimulator
from src.payoffs import VanillaCall
from src.analytical import black_scholes, compute_error_metrics

plt.style.use('seaborn-v0_8-whitegrid')
FIGSIZE = (10, 5)  # consistent figure size
print('Imports OK')

## 1  Configuration & Black-Scholes Baseline

In [ ]:
# Typical AAPL-like ATM parameters
BASE_CFG = SimulatorConfig(
    S0=100.0, K=100.0, T=1.0, r=0.05, sigma=0.2,
    n_paths=100_000, n_steps=252, seed=42
)

BS_CALL = black_scholes(BASE_CFG, option_type='call')
print(f'Black-Scholes call price : ${BS_CALL:.6f}')

## 2  Plot 1 — Sample Path Trajectories

In [ ]:
N_DISPLAY = 100  # paths to draw
display_cfg = SimulatorConfig(
    S0=BASE_CFG.S0, K=BASE_CFG.K, T=BASE_CFG.T,
    r=BASE_CFG.r, sigma=BASE_CFG.sigma,
    n_paths=N_DISPLAY, n_steps=BASE_CFG.n_steps, seed=7
)
sim_display = GBMSimulator(display_cfg)
paths = sim_display.simulate_paths(method='standard').cpu().numpy()  # (100, n_steps+1)

t_grid = np.linspace(0, display_cfg.T, display_cfg.n_steps + 1)
path_mean = paths.mean(axis=0)
path_std  = paths.std(axis=0)

fig, ax = plt.subplots(figsize=FIGSIZE)
for i in range(N_DISPLAY):
    ax.plot(t_grid, paths[i], color='steelblue', alpha=0.15, linewidth=0.6)

ax.plot(t_grid, path_mean, color='crimson', linewidth=2.5, label='Mean path', zorder=5)
ax.fill_between(
    t_grid,
    path_mean - 1.96 * path_std,
    path_mean + 1.96 * path_std,
    color='crimson', alpha=0.15, label='±1.96σ (95% CI)'
)

ax.axhline(display_cfg.K, color='black', linestyle='--', linewidth=1, label=f'Strike K={display_cfg.K}')
ax.set_xlabel('Time (years)', fontsize=12)
ax.set_ylabel('Asset Price $S_t$', fontsize=12)
ax.set_title(f'GBM Path Trajectories (N={N_DISPLAY}), σ={display_cfg.sigma}, r={display_cfg.r}', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../notebooks/plot_paths.png', dpi=150)
plt.show()

## 3  Convergence Study — Sweep N

In [ ]:
N_SWEEP = [100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]
METHODS = ['standard', 'antithetic', 'sobol']
COLORS  = {'standard': '#1f77b4', 'antithetic': '#ff7f0e', 'sobol': '#2ca02c'}

results = {m: [] for m in METHODS}

for n in N_SWEEP:
    cfg = SimulatorConfig(
        S0=BASE_CFG.S0, K=BASE_CFG.K, T=BASE_CFG.T,
        r=BASE_CFG.r, sigma=BASE_CFG.sigma,
        n_paths=n, n_steps=BASE_CFG.n_steps, seed=42
    )
    sim = GBMSimulator(cfg)
    payoff = VanillaCall(cfg.K)
    for m in METHODS:
        res = sim.price(payoff, method=m)
        results[m].append(res['price'])
    print(f'N={n:>7,d}  ', '  '.join(f"{m}={results[m][-1]:.4f}" for m in METHODS))

print(f'\nBS reference: {BS_CALL:.4f}')

## 4  Plot 2 — Convergence Curve

In [ ]:
fig, ax = plt.subplots(figsize=FIGSIZE)

for m in METHODS:
    ax.plot(N_SWEEP, results[m], marker='o', markersize=5, label=m.capitalize(), color=COLORS[m])

ax.axhline(BS_CALL, color='black', linestyle='--', linewidth=1.5, label=f'Black-Scholes ({BS_CALL:.4f})')

ax.set_xscale('log')
ax.set_xlabel('Number of Paths (N)', fontsize=12)
ax.set_ylabel('Option Price Estimate', fontsize=12)
ax.set_title('MC Convergence: Standard vs Antithetic vs Sobol', fontsize=13)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../notebooks/plot_convergence.png', dpi=150)
plt.show()

## 5  Plot 3 — Terminal Price Distribution

In [ ]:
# Use a large simulation to get a smooth histogram
hist_cfg = SimulatorConfig(
    S0=BASE_CFG.S0, K=BASE_CFG.K, T=BASE_CFG.T,
    r=BASE_CFG.r, sigma=BASE_CFG.sigma,
    n_paths=100_000, n_steps=BASE_CFG.n_steps, seed=0
)
sim_hist = GBMSimulator(hist_cfg)
paths_hist = sim_hist.simulate_paths(method='standard').cpu().numpy()
S_T = paths_hist[:, -1]  # terminal prices

# Log-normal fit parameters: mean and std of log(S_T)
mu_log = math.log(hist_cfg.S0) + (hist_cfg.r - 0.5 * hist_cfg.sigma**2) * hist_cfg.T
sigma_log = hist_cfg.sigma * math.sqrt(hist_cfg.T)

x_range = np.linspace(S_T.min(), S_T.max(), 500)
lognormal_pdf = lognorm.pdf(x_range, s=sigma_log, scale=math.exp(mu_log))

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.hist(S_T, bins=100, density=True, color='steelblue', alpha=0.65, label='MC $S_T$ histogram')
ax.plot(x_range, lognormal_pdf, color='crimson', linewidth=2.5, label='Log-Normal PDF')
ax.axvline(hist_cfg.K, color='black', linestyle='--', linewidth=1.5, label=f'Strike K={hist_cfg.K}')

ax.set_xlabel('Terminal Asset Price $S_T$', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Distribution of Terminal Asset Prices $S_T$ with Log-Normal Overlay', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../notebooks/plot_distribution.png', dpi=150)
plt.show()

## 6  Error Metrics at N = 100,000

In [ ]:
import pandas as pd

rows = []
for m in METHODS:
    mc_price = results[m][-1]  # last entry = N=100_000
    metrics = compute_error_metrics(mc_price, BS_CALL)
    rows.append({
        'Method': m.capitalize(),
        'MC Price': f'{mc_price:.6f}',
        'BS Price': f'{BS_CALL:.6f}',
        'MSE': f'{metrics["mse"]:.2e}',
        'APE (%)': f'{metrics["ape"]:.4f}',
    })

df = pd.DataFrame(rows).set_index('Method')
print(df.to_string())